Lấy 3 chỉ số chứng khoán, so sánh mức độ tăng giảm so với ngày hôm trước

In [3]:
from vnstock import Quote
import pandas as pd
from datetime import timedelta,time

symbols = ['VNINDEX', 'VN30', 'HNX30']
data_frames = []

for sym in symbols:
    quote = Quote(symbol=sym, source='VCI')
    df = quote.history(length='1', interval='d').reset_index()  # reset index để có cột time thay vì index
    # chèn cột mới ở vị trí 0 (đầu DataFrame)
    df.insert(0, 'symbol', sym)
    # df = df.df[['symbol', 'time', 'close', 'volume']]
    # trong df, lấy ra các cột cần thiết: symbol, time, close, volume
    df = df[['symbol', 'time', 'close', 'volume']]
    data_frames.append(df)
    
    # print(f"Data for {sym}:\n{df}\n")

# nếu muốn gộp toàn bộ thành một bảng duy nhất:
# all_data_sorted = pd.concat(data_frames, ignore_index=True).sort_values(by=['symbol', 'date'])
all_data = pd.concat(data_frames, ignore_index=True)
# all_data = pd.concat(data_frames, ignore_index=True)
# all_data
# từ bảng all_data
# đẩy giá của ngày 27/2 xuống ngày 2/3
# so sánh thay đổi điểm và phần trăm thay đổi

# all_data['time'] = pd.to_datetime(all_data['time'])

# trong từng symsbol, lấy giá của ngày d-1, gán vào ngày d-2, tính chênh lệch điểm và phần trăm thay đổi
all_data['prev_close'] = all_data.groupby('symbol')['close'].shift(1)
all_data['change'] = all_data['close'] - all_data['prev_close']
all_data['change_pct'] = (all_data['change'] / all_data['prev_close'] * 100).round(2)	

today = pd.Timestamp.now().date().strftime('%Y-%m-%d')
all_data_today = all_data[all_data['time'] == today]

all_data_today.reset_index()
# all_data.to_csv('index_data.csv', index=False)

,index,symbol,time,close,volume,prev_close,change,change_pct
0,2,VNINDEX,2026-03-13,1696.24,1023068442,1709.61,-13.37,-0.78
1,5,VN30,2026-03-13,1853.60,372106248,1859.80,-6.20,-0.33
2,8,HNX30,2026-03-13,524.78,94164714,532.70,-7.92,-1.49


Non-200 response fetching content: 400


In [4]:
all_data_today.to_csv('index_data.csv', index=False)

# 2. lấy top 10 cổ phiếu vốn hóa giảm

In [ ]:
# code lấy top 10 cổ phiếu có vốn hóa giảm/tăng
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
# df =["STG.VN","ACB.VN", "VCB.VN","SSI.VN","FPT.VN","GAS.VN","MSN.VN","VIC.VN","VNM.VN","VPB.VN","CTG.VN","TCB.VN","DGW.VN","FRT.VN"]
vn30 = pd.read_csv('ticker_vn30.csv')           # hoặc Listing(...).all_symbols()
vn30 = vn30['symbol'].astype(str) + ".VN"  # thêm ".VN" vào mỗi mã để phù hợp với yfinance
# symbols = symbols.tolist()  # chuyển cột 'symbol' thành list

start=(datetime.today() - timedelta(days=2)).strftime("%Y-%m-%d")
end= datetime.today().strftime("%Y-%m-%d")
#trong yfinance, tối muốn lấy giá đóng cửa của ngày start và end, từ đó tính ra thay đổi điểm và phần trăm thay đổi của cổ phiếu trong khoảng thời gian đó
# trình bày kết quả dưới dạng bảng với các cột: Ticker, Close End, Change Points, Change Percent
records = []
for ticker in vn30:
	data = yf.download(ticker, start=start, end=end)
	sharesOutstanding = yf.Ticker(ticker).info.get('sharesOutstanding', 'N/A')
	if data.empty:
		print(f"No data for {ticker} between {start} and {end}.")
		continue
	close_start = data['Close'].iloc[0]
	close_end = data['Close'].iloc[-1]
	sharesOutstanding = sharesOutstanding
	change_points = close_end - close_start
	change_percent = (change_points / close_start)*100
	market_cap_d_2ays_ago = (close_start * sharesOutstanding)/1e9
	maket_cap_now = (close_end * sharesOutstanding)/1e9
	change_market_cap_bil_vnd = maket_cap_now - market_cap_d_2ays_ago
	records.append({
		'Ticker': ticker.replace(".VN", "")
		,'Close End': close_end
		,'Change Points': change_points
		,'Change Percent': change_percent
		,'Market Cap 2 Days Ago (Bil VND)': market_cap_d_2ays_ago
		,'Market Cap Now (Bil VND)': maket_cap_now
		,'Change Market Cap (Bil VND)': change_market_cap_bil_vnd
		})
# trình bày kết quả dưới dạng bảng với các cột: Ticker, Close End, Change Points, Change Percent
# trong cell dữ liệu của close_end, change point, change percent, chỉ lấy ra số liệu
records= pd.DataFrame(records)
# Chuyển các cột về kiểu float để chỉ hiển thị số
records['Close End'] = records['Close End'].astype(float)
records['Change Points'] = records['Change Points'].astype(float)
records['Change Percent'] = records['Change Percent'].astype(float)
records['Market Cap 2 Days Ago (Bil VND)'] = records['Market Cap 2 Days Ago (Bil VND)'].astype(float)
records['Market Cap Now (Bil VND)'] = records['Market Cap Now (Bil VND)'].astype(float)
records['Change Market Cap (Bil VND)'] = records['Change Market Cap (Bil VND)'].astype(float)
# records['sharesOutstanding'] = records['Shares Outstanding'].astype(int)
records.sort_values(by='Change Market Cap (Bil VND)', ascending=False, inplace=True)
records.reset_index(drop=True, inplace=True)
records

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Ticker,Close End,Change Points,Change Percent,Market Cap 2 Days Ago (Bil VND),Market Cap Now (Bil VND),Change Market Cap (Bil VND)
0,VHM,98000.0,2100.0,0.021898,3.939008e+05,4.025264e+05,8625.565208
1,VNM,63100.0,1500.0,0.024351,1.287413e+05,1.318762e+05,3134.933168
2,VJC,156800.0,4000.0,0.026178,9.039821e+04,9.276466e+04,2366.445336
3,ACB,23450.0,300.0,0.012959,1.189136e+05,1.204546e+05,1540.996980
4,LPB,41500.0,400.0,0.009732,1.227773e+05,1.239722e+05,1194.912840
5,MSN,73900.0,800.0,0.010944,1.056964e+05,1.068532e+05,1156.732366
6,VIB,16900.0,250.0,0.015015,5.667670e+04,5.752770e+04,851.001427
7,SAB,44400.0,650.0,0.014857,5.611210e+04,5.694577e+04,833.665542
8,STB,65800.0,100.0,0.001522,1.238587e+05,1.240472e+05,188.521572
9,SSB,16550.0,0.0,0.000000,4.708475e+04,4.708475e+04,0.000000


In [27]:
records.to_csv('top_10_high.csv')

3. get stock high performance

In [ ]:
df = pd.read_csv('ticker_hose.csv')           # hoặc Listing(...).all_symbols()
df = df['symbol'].astype(str) + ".VN"
df.head()

In [62]:
scanner = []
for ticker in df:
    info = yf.Ticker(ticker).info
    price = info.get('regularMarketPrice', '')
    ROE = info.get('returnOnEquity', '')*100
    ROA = info.get('returnOnAssets', '')*100
    ROIC = info.get('returnOnInvestedCapital', '')
    PE = info.get('trailingPE', '')
    PB = info.get('priceToBook','')
    EPS = info.get('trailingEps','')
    
    scanner.append({
    'Ticker': ticker.replace(".VN", "")
    ,'Price': price
    ,'ROE': ROE
    ,'ROA': ROA
    ,'ROIC':ROIC
    ,'PE': PE
    ,'PB': PB
    ,'EPS': EPS

  })
scanner_df = pd.DataFrame(scanner)

# Convert ranking fields to numeric (some tickers may return non-numeric placeholders like '' or 'N/A')
for col in ['ROE','ROA','ROIC','PE','PB','EPS']:
    scanner_df[col] = pd.to_numeric(scanner_df[col], errors='coerce')


In [63]:

# Tạo cột xếp hạng roe, roa, pe, pb
scanner_df['ROE_Rank'] = scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom')
# scanner_df['ROA_Rank'] = scanner_df['ROA'].rank(method='dense',ascending=False, na_option='bottom')
scanner_df['PE_Rank'] = scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')
scanner_df['ROE + PE'] = (scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom') 
                          + 
                          (scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')))
scanner_df['ROE + PE Ranking'] = scanner_df['ROE + PE'].rank(method='dense',ascending=True, na_option='bottom').astype(int)
scanner_df.sort_values(by='ROE + PE Ranking',ascending=True).reset_index()

,index,Ticker,Price,ROE,ROA,ROIC,PE,PB,EPS,ROE_Rank,PE_Rank,ROE + PE,ROE + PE Ranking
0,89,DLG,2630.0,47.737998,5.774,NaN,2.157488,0.821552,1219.01,6.0,2.0,8.0,1
1,360,VCG,23500.0,34.862998,2.799,NaN,3.862495,1.309214,6084.15,12.0,3.0,15.0,2
2,147,HHS,11200.0,33.600000,-0.148,NaN,1.208313,0.535221,9269.12,15.0,1.0,16.0,3
3,275,SCS,52900.0,50.620000,26.762,NaN,6.684492,3.222240,7913.84,3.0,18.0,21.0,4
4,214,NCT,92400.0,61.089003,31.254,NaN,6.849315,3.404071,13490.40,2.0,20.0,22.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,11,ADP,23200.0,NaN,NaN,NaN,NaN,1.906283,NaN,339.0,184.0,523.0,273
397,9,ACL,13550.0,NaN,NaN,NaN,NaN,0.816043,NaN,339.0,184.0,523.0,273
398,46,CCI,27000.0,NaN,NaN,NaN,NaN,1.654946,NaN,339.0,184.0,523.0,273
399,398,VVS,160500.0,NaN,NaN,NaN,NaN,NaN,NaN,339.0,184.0,523.0,273


In [54]:
scanner_df.to_csv('ranking_ticker.csv')

3. Lấy 3 bảng: income_statement, balance_sheet, valuation_measures

In [ ]:
def get_income_statement(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    stmt = yf.Ticker(ticker).get_income_stmt(freq='quarterly')
    df = pd.DataFrame(stmt).loc[['TotalRevenue', 'CostOfRevenue', 'GrossProfit', 'NetIncomeContinuousOperations']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'TotalRevenue': df.at['TotalRevenue', date_col] / 1e9,
            'CostOfRevenue': df.at['CostOfRevenue', date_col] / 1e9,
            'GrossProfit': df.at['GrossProfit', date_col] / 1e9,
            'NetIncome': df.at['NetIncomeContinuousOperations', date_col] / 1e9,
        }
        row['GrossProfitMargin'] = 100.0*round(row['GrossProfit'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        row['NetIncomeMargin'] = 100.0*round(row['NetIncome'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_income_statements(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_income_statement(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_stmt = get_income_statements(tickers)
merged_stmt

OK: AAA.VN
SKIP: AAM.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetIncomeContinuousOperations'],\n      dtype='object')] are in the [index]"
SKIP: AAT.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetIncomeContinuousOperations'],\n      dtype='object')] are in the [index]"
SKIP: ABR.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetIncomeContinuousOperations'],\n      dtype='object')] are in the [index]"
SKIP: ABS.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetIncomeContinuousOperations'],\n      dtype='object')] are in the [index]"
SKIP: ABT.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetIncomeContinuousOperations'],\n      dtype='object')] are in the [index]"
SKIP: ACB.VN - "['CostOfRevenue', 'GrossProfit'] not in index"
SKIP: ACC.VN - "None of [Index(['TotalRevenue', 'CostOfRevenue', 'GrossProfit',\n       'NetInco

,Ticker,Date,TotalRevenue,CostOfRevenue,GrossProfit,NetIncome,GrossProfitMargin,NetIncomeMargin
0,AAA,2024-06-30,NaN,NaN,NaN,NaN,NaN,NaN
1,AAA,2024-09-30,3193.399535,2848.063507,345.336028,-25.701898,0.11,-0.01
2,AAA,2024-12-31,3842.519613,3393.914631,448.604982,63.675468,0.12,0.02
3,AAA,2025-03-31,3856.408102,3392.693587,463.714514,55.512421,0.12,0.01
4,AAA,2025-06-30,2309.974268,1955.834761,354.139506,171.766008,0.15,0.07
...,...,...,...,...,...,...,...,...
964,YEG,2024-12-31,397.024367,336.979053,60.045315,66.823387,0.15,0.17
965,YEG,2025-03-31,217.744210,174.479831,43.264379,23.252972,0.20,0.11
966,YEG,2025-06-30,455.711172,400.724449,54.986723,33.298712,0.12,0.07
967,YEG,2025-09-30,390.953829,345.662574,45.291256,7.342328,0.12,0.02


In [56]:
merged_stmt = merged_stmt.dropna(subset=['TotalRevenue','CostOfRevenue','GrossProfit','NetIncome','GrossProfitMargin','NetIncomeMargin'])


In [57]:
merged_stmt.to_csv('income_statement.csv')

In [58]:
def get_balance_sheet(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    bls = yf.Ticker(ticker).get_balance_sheet(freq='quarterly')
    df = pd.DataFrame(bls).loc[['TotalAssets','TotalLiabilitiesNetMinorityInterest','TotalEquityGrossMinorityInterest']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'Total Assets': df.at['TotalAssets', date_col] / 1e9,
            'Total Liabilities': df.at['TotalLiabilitiesNetMinorityInterest', date_col] / 1e9,
            'Total Equity': df.at['TotalEquityGrossMinorityInterest', date_col] / 1e9,
        }
        row['Liabilities/Equity ratio'] = round(row['Total Liabilities'] / row['Total Equity'], 2) if row['Total Equity'] else None

        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_balance_sheets(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_balance_sheet(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_balance_sheet = get_balance_sheets(tickers)
merged_balance_sheet.dropna(subset= ['Total Assets','Total Liabilities','Total Equity','Liabilities/Equity ratio'])

OK: AAA.VN
SKIP: AAM.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEquityGrossMinorityInterest'],\n      dtype='object')] are in the [index]"
SKIP: AAT.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEquityGrossMinorityInterest'],\n      dtype='object')] are in the [index]"
SKIP: ABR.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEquityGrossMinorityInterest'],\n      dtype='object')] are in the [index]"
SKIP: ABS.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEquityGrossMinorityInterest'],\n      dtype='object')] are in the [index]"
SKIP: ABT.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEquityGrossMinorityInterest'],\n      dtype='object')] are in the [index]"
OK: ACB.VN
SKIP: ACC.VN - "None of [Index(['TotalAssets', 'TotalLiabilitiesNetMinorityInterest',\n       'TotalEqu

,Ticker,Date,Total Assets,Total Liabilities,Total Equity,Liabilities/Equity ratio
1,AAA,2024-09-30,13032.239744,6955.710255,6076.529488,1.14
2,AAA,2024-12-31,13768.215584,7531.941631,6236.273953,1.21
3,AAA,2025-03-31,12184.753379,6147.473304,6037.280075,1.02
4,AAA,2025-06-30,12183.297027,6278.671129,5904.625898,1.06
6,AAA,2025-12-31,12891.604734,6812.374374,6079.230360,1.12
...,...,...,...,...,...,...
1183,YEG,2024-12-31,2512.869601,1012.991102,1499.878499,0.68
1184,YEG,2025-03-31,2571.459400,500.322520,2071.136881,0.24
1185,YEG,2025-06-30,2684.515872,584.256179,2100.259693,0.28
1186,YEG,2025-09-30,2753.758052,643.424347,2110.333705,0.30


In [59]:
merged_balance_sheet.to_csv('balance_sheet.csv')

4. lấy giá và sma20

In [60]:
import yfinance as yf
import pandas as pd
import talib
from datetime import datetime,timedelta


start = ((datetime.today() - timedelta(days=120)).replace(day=1).strftime("%Y-%m-%d"))
end = datetime.today().strftime("%Y-%m-%d")

def get_price(ticker):
    """Get price history for a single ticker, return DataFrame with Ticker and Date columns."""
    hist = yf.download(ticker, start=start, end=end, interval='1d')
    close = hist['Close'].reset_index()
    df = close.melt(var_name='Ticker', id_vars='Date', value_name='Close')
    df = df.sort_values(by=['Ticker', 'Date'])
    df['Ticker'] = df['Ticker'].str.replace('.VN', '')
    df['SMA20'] = talib.SMA(df['Close'], timeperiod=20)
    return df

def get_prices(tickers):
    all_hist = []
    for ticker in tickers:
        get_hist = get_price(ticker)
        all_hist.append(get_hist)
    result = pd.concat(all_hist).sort_values(by=['Ticker','Date']).reset_index(drop=True)
    return result

merged = get_prices(df)
merged

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Date,Ticker,Close,SMA20
0,2025-11-03,AAA,8000.0,NaN
1,2025-11-04,AAA,8120.0,NaN
2,2025-11-05,AAA,8070.0,NaN
3,2025-11-06,AAA,8120.0,NaN
4,2025-11-07,AAA,7950.0,NaN
...,...,...,...,...
33564,2026-03-09,YEG,10850.0,12157.5
33565,2026-03-10,YEG,10750.0,12077.5
33566,2026-03-11,YEG,11050.0,12000.0
33567,2026-03-12,YEG,10900.0,11930.0


In [61]:
merged.to_csv('hist_price_data.csv')